# VoxIntel — 05 Debugging: Fine-tune Wav2Vec2 on SLURP (Fixed Run)

## Why the previous `05_finetune_wav2vec2.ipynb` run failed

The previous run had `SMOKE_TEST = False` and `NUM_EPOCHS = 15` correctly
set, so on paper it should have been a full training run. But looking at
the actual training log, it stopped after **2,000 / 94,920 steps — about
0.32 epochs** — and the final validation WER (0.967) barely differs from
notebook 04's *pretrained, non-fine-tuned* baseline (0.984 corpus WER).

The root cause is visible directly in the logged eval metrics:

| Step | Training Loss | Eval WER | Eval CER |
|-----:|---------------:|---------:|---------:|
| 500  | 3.138           | 0.956482 | 0.966406 |
| 1000 | 1.802           | 0.956482 | 0.966406 |
| 1500 | 1.667           | 0.956482 | 0.966406 |
| 2000 | 1.593           | 0.956482 | 0.966406 |

Training loss was falling steadily (3.14 → 1.59), which means the model
**was** learning. But `eval_wer` was *bit-for-bit identical* across all
four eval checkpoints. That is not a coincidence — it is the classic CTC
**blank-collapse** phase: early in CTC training, the model routes almost
all probability mass to the blank token and predicts an empty (or
near-empty) transcript for everything, which produces a constant,
maximal WER regardless of small improvements in the underlying logits.
The model needs enough gradient steps to escape this regime before WER
starts moving at all.

`EarlyStoppingCallback(patience=3)` was watching `eval_wer`, saw the
*exact same* "best" value four times in a row, concluded there was no
further improvement possible, and killed training — right in the middle
of the collapse phase, before the model had any chance to escape it.
Two compounding factors made the collapse phase last longer than it
needed to:

- **`LR = 1e-4`** is aggressive for Wav2Vec2 CTC fine-tuning (commonly
  1e-5–5e-5 range), which can make early optimization less stable and
  slower to leave the blank-collapse basin.
- **`WARMUP_STEPS = 500`** is a fixed step count that, relative to a
  94,920-step schedule, is a very short (~0.5%) ramp — the learning rate
  reached full value almost immediately, rather than easing the model
  through the unstable early phase.

So: **the pipeline itself (data loading, collator, label alignment, CTC
validity filtering) is correct** — training loss decreasing proves that.
The failure was a *training-dynamics* bug: early stopping fired on a
metric that hadn't started moving yet, for a completely explainable
reason, and the aggressive LR/warmup combination made that window wider
than it needed to be.

## What's new in this notebook

Based on that diagnosis, plus the tuning pass previously done on this
notebook, this version changes:

| Change | Why |
|---|---|
| `LR: 1e-4 → 3e-5` | Standard Wav2Vec2 CTC fine-tuning range; less likely to destabilize early training |
| `warmup_ratio = 0.1` (was fixed `warmup_steps=500`) | Scales the ramp to the *actual* full-dataset schedule instead of a fixed, now-tiny fraction of it |
| **Delayed early stopping** (new custom callback) | Ignores `eval_wer` entirely for the first `MIN_STEPS_BEFORE_EARLY_STOPPING` steps, so blank-collapse can't be mistaken for convergence |
| `EARLY_STOPPING_PATIENCE: 3 → 10` | Extra margin per your choice, on top of the delay guard above |
| `lr_scheduler_type = "cosine"` | Smoother decay than the default linear schedule |
| `weight_decay = 0.01` | Was missing; standard regularization for this setup |
| `group_by_length = True` | Batches similarly-sized audio together — less wasted padding, more stable batch statistics |
| `gradient_checkpointing = True` | Trades compute for VRAM — needed headroom for the 6–8GB GPU this notebook is configured for |
| `fp16 = True` | Mixed precision — roughly halves activation memory, important at 6–8GB VRAM |
| `GRADIENT_ACCUMULATION_STEPS: 2 → 4` (effective batch 4×4=16, same per-step batch of 4) | Smoother gradients without raising per-step VRAM use |
| `EVAL_SUBSET_SIZE_DURING_TRAINING: 500 → 1000` | You said time isn't a constraint — a larger training-time eval subset gives a less noisy WER signal for checkpoint selection |
| `SMOKE_TEST = False`, full `train` split, full `NUM_EPOCHS = 15` | Per your instruction — this run trains on the complete dataset, no shrinking |

**`FREEZE_FEATURE_ENCODER` stays `True` for this run.** Unfreezing the
CNN feature encoder is a real lever (Strategy 2 in the earlier tuning
notes) but it's deliberately *not* mixed in here: this notebook changes
the training-dynamics bug and applies safe, standard hyperparameters in
one pass. Unfreezing is a separate, independent experiment — bundling it
in here would make it impossible to tell whether an improvement came
from fixing the bug or from unfreezing. Treat that as `05b` if you want
to run it next, after this run establishes a real fine-tuned baseline.

## A note on "perfect scores"

Genuinely perfect (0% WER) transcription isn't a realistic target for
any ASR system, including commercial ones — SLURP has noisy, accented,
naturalistic home-assistant audio, and some residual error rate is
expected even from strong models. What this notebook is built to do is
**get the model out of the blank-collapse trap and into a real training
regime**, so the WER you see actually reflects the model's fine-tuned
performance instead of an early-stopping artifact. Realistically, a
successful run here should land somewhere in the 15–35% WER range
depending on how far training gets — a large, real improvement over the
96.7%/58.4% baseline numbers, not a literal zero.


Make `src` importable, same as notebooks 03/04.

In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\ACER\OneDrive\Desktop\VoxIntel


## Imports

In [6]:
import json
import random
import dataclasses
from typing import Any, Dict, List, Union

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import evaluate
from datasets import Dataset
from transformers import (
    AutoProcessor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

from src.data import SLURPDataset
from src.utils.audio import load_audio
from src.utils.text import normalize_text

## Config

Everything that controls this run lives here. Compared to the previous
notebook, the training-dynamics parameters (`LR`, `WARMUP_RATIO`,
`EARLY_STOPPING_PATIENCE`, `MIN_STEPS_BEFORE_EARLY_STOPPING`) are the
ones responsible for actually fixing the bug — the rest are efficiency
and stability additions layered on top.


In [ ]:
MODEL_NAME = "facebook/wav2vec2-base-960h"
ROOT_DIR = "C:\\Users\\ACER\\OneDrive\\Desktop\\VoxIntel\\data\\raw\\slurp"
TRAIN_SPLIT = "train"
VAL_SPLIT = "validation"
OUTPUT_DIR = PROJECT_ROOT / "models" / "wav2vec2_slurp_debug"
SEED = 42

# Full dataset, full run. No shrinking.
SMOKE_TEST = False

LR = 3e-5                     # was 1e-4 -- standard Wav2Vec2 CTC fine-tuning range
WARMUP_RATIO = 0.1            # was fixed WARMUP_STEPS=500 (~0.5% of the real schedule)
LR_SCHEDULER_TYPE = "cosine"  # smoother decay than default linear

# Early stopping fix 
# The previous run's early stopping fired while eval_wer was frozen during
# CTC blank-collapse (see markdown above). Two independent guards now:
#   1) a hard "don't even look at eval_wer" window early in training, and
#   2) a larger patience on top of that, per your preference.
MIN_STEPS_BEFORE_EARLY_STOPPING = 3000
EARLY_STOPPING_PATIENCE = 10

BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4   # effective batch = 16
GRADIENT_CHECKPOINTING = True
FP16 = True
GROUP_BY_LENGTH = True
WEIGHT_DECAY = 0.01

NUM_EPOCHS = 15
EVAL_STEPS = 500
SAVE_STEPS = 500

MAX_AUDIO_DURATION_SEC = 15
EVAL_SUBSET_SIZE_DURING_TRAINING = 1000   # was 500 -- time isn't a constraint here
FREEZE_FEATURE_ENCODER = True             # unchanged deliberately -- see markdown above

## Seed and device

In [8]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("WARNING: no CUDA device found. This config assumes a 6-8GB GPU; "
          "training on CPU will be extremely slow for the full dataset.")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Total VRAM (GB): 4.29


## Load train and validation splits

Real audio only for both splits, same as before. `train_synthetic` is
still left out — that stays a follow-up experiment, not something to mix
in while we're isolating the training-dynamics fix.


In [9]:
train_set = SLURPDataset(split=TRAIN_SPLIT, root_dir=ROOT_DIR)
val_set = SLURPDataset(split=VAL_SPLIT, root_dir=ROOT_DIR)

print(train_set.summary())
print(val_set.summary())

{'split': 'train', 'n_samples': 50628, 'n_unique_intents': 91, 'n_missing_audio': 0, 'n_missing_transcript': 0, 'n_missing_intent': 0}
{'split': 'validation', 'n_samples': 8690, 'n_unique_intents': 71, 'n_missing_audio': 0, 'n_missing_transcript': 0, 'n_missing_intent': 0}


## Convert to plain rows

Same filtering as before: drop missing audio, empty transcript, or audio
over `MAX_AUDIO_DURATION_SEC`. The duration cutoff still matters for CTC
validity (see the next filtering step below) as much as for efficiency.


In [10]:
def to_rows(dataset, max_duration=MAX_AUDIO_DURATION_SEC):
    rows = []
    skipped = {"no_audio": 0, "no_transcript": 0, "too_long": 0, "unreadable": 0}

    for sample in dataset:
        if not sample["audio_path"]:
            skipped["no_audio"] += 1
            continue
        if not sample["transcript"] or not sample["transcript"].strip():
            skipped["no_transcript"] += 1
            continue

        try:
            waveform, sr = load_audio(sample["audio_path"], target_sr=16_000)
        except Exception:
            skipped["unreadable"] += 1
            continue

        duration = len(waveform) / sr
        if duration > max_duration:
            skipped["too_long"] += 1
            continue

        rows.append({
            "audio_path": str(sample["audio_path"]),
            "sentence": sample["transcript"],
            "intent": sample["intent"],
            "scenario": sample["scenario"],
        })

    print(f"Kept {len(rows)} / {len(dataset)} samples. Skipped: {skipped}")
    return rows


train_rows = to_rows(train_set)
val_rows = to_rows(val_set)

Kept 50619 / 50628 samples. Skipped: {'no_audio': 0, 'no_transcript': 0, 'too_long': 9, 'unreadable': 0}
Kept 8690 / 8690 samples. Skipped: {'no_audio': 0, 'no_transcript': 0, 'too_long': 0, 'unreadable': 0}


## Convert to Hugging Face Datasets

In [2]:
train_hf = Dataset.from_list(train_rows)
val_hf = Dataset.from_list(val_rows)

train_hf, val_hf

NameError: name 'Dataset' is not defined

## Smoke-test subset (disabled)

`SMOKE_TEST = False` for this run, per your instruction — the full
train and validation splits are used, no shrinking.


In [17]:
if SMOKE_TEST:
    train_hf = train_hf.select(range(min(200, len(train_hf))))
    val_hf = val_hf.select(range(min(50, len(val_hf))))

print(f"train_hf: {len(train_hf)} rows, val_hf: {len(val_hf)} rows")

train_hf: 50619 rows, val_hf: 8690 rows


## Load processor

In [18]:
processor = AutoProcessor.from_pretrained(MODEL_NAME)

## Preprocessing function

Unchanged from the previous notebook: builds `input_values` from the
resampled waveform and `labels` from the uppercased, normalized
transcript (this checkpoint's tokenizer vocabulary is uppercase).


In [19]:
import numpy as np

def preprocess_function(batch):
    waveform, sr = load_audio(batch["audio_path"], target_sr=16_000)

    input_values = np.asarray(
        processor(
            waveform,
            sampling_rate=sr
        ).input_values[0],
        dtype=np.float32
    )

    labels = processor(
        text=normalize_text(batch["sentence"]),
        return_attention_mask=False,
    ).input_ids

    return {
        "input_values": input_values,
        "input_length": len(input_values),
        "labels": labels,
    }

## Map preprocessing over both splits

In [20]:
train_proc = train_hf.map(
    preprocess_function,
    remove_columns=train_hf.column_names,
    writer_batch_size=2,
    keep_in_memory=False,
)

val_proc_full = val_hf.map(
    preprocess_function,
    remove_columns=val_hf.column_names,
    writer_batch_size=2,
    keep_in_memory=False,
)

Map:  23%|██▎       | 11549/50619 [03:18<11:10, 58.27 examples/s] 


ArrowMemoryError: realloc of size 4294967296 failed

## Drop CTC-invalid examples

Unchanged: Wav2Vec2's conv feature extractor downsamples audio by
roughly 320x, so the resulting frame count must be at least as long as
the label sequence for CTC loss to be defined.


In [ ]:
def is_ctc_valid(example, downsample_factor=320):
    output_length = example["input_length"] // downsample_factor
    return output_length >= len(example["labels"])


before = len(train_proc)
train_proc = train_proc.filter(is_ctc_valid)
print(f"train_proc: kept {len(train_proc)} / {before}")

before = len(val_proc_full)
val_proc_full = val_proc_full.filter(is_ctc_valid)
print(f"val_proc_full: kept {len(val_proc_full)} / {before}")

## Two-tier validation set

`val_proc_train` is now **1,000** examples (up from 500) — a larger,
less noisy training-time eval subset, since the noisy 500-example signal
was part of what made it hard to tell blank-collapse apart from genuine
non-improvement. `val_proc_full` (all of validation) is still reserved
for the one full pass at the end, directly comparable to notebook 04's
baseline.


In [ ]:
eval_subset_size = min(EVAL_SUBSET_SIZE_DURING_TRAINING, len(val_proc_full))
val_proc_train = val_proc_full.select(range(eval_subset_size))

print(f"val_proc_train (used during training): {len(val_proc_train)}")
print(f"val_proc_full (used for final evaluation): {len(val_proc_full)}")

## Data collator for CTC

Unchanged: standard Wav2Vec2 CTC collator, pads `input_values` and
`labels` separately and replaces label padding with -100.


In [ ]:
@dataclasses.dataclass
class DataCollatorCTCWithPadding:
    processor: Any
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):

        input_features = [
            {"input_values": f["input_values"]}
            for f in features
        ]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": f["labels"]}
            for f in features
        ]

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["attention_mask"].ne(1),
            -100,
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(processor)

## Evaluation metric function

Unchanged: WER and CER computed on decoded predictions during training.
This is the metric that was frozen during blank-collapse in the previous
run — the fix here is in when/how it's allowed to trigger early stopping,
not in how it's computed.


In [ ]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

## Load pretrained model

`FREEZE_FEATURE_ENCODER` stays on, same reasoning as before: the CNN
feature encoder was learned on 960 hours of clean audiobook audio, and
freezing it protects those low-level acoustic features while the rest
of the model adapts to SLURP.

New here: `model.gradient_checkpointing_enable()`, needed to keep memory
usage inside a 6-8GB budget with `fp16` training. `ctc_zero_infinity=True`
is unchanged — extra safety against any remaining frame-length-vs-label
mismatch.


In [ ]:
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

if FREEZE_FEATURE_ENCODER:
    model.freeze_feature_encoder()

if GRADIENT_CHECKPOINTING:
    model.gradient_checkpointing_enable()
    model.config.use_cache = False  # required whenever gradient checkpointing is on

model.to(DEVICE)

## Delayed early stopping callback

This is the core fix for the bug diagnosed above. A plain
`EarlyStoppingCallback` starts counting non-improving evals from step 1
— which is exactly what killed the previous run during blank-collapse.
This subclass ignores evaluation results entirely until
`MIN_STEPS_BEFORE_EARLY_STOPPING` has passed, and only then hands control
to the normal `EarlyStoppingCallback` logic with `EARLY_STOPPING_PATIENCE`
patience.


In [ ]:
class DelayedEarlyStoppingCallback(EarlyStoppingCallback):
    def __init__(self, min_steps_before_stopping, **kwargs):
        super().__init__(**kwargs)
        self.min_steps_before_stopping = min_steps_before_stopping

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if state.global_step < self.min_steps_before_stopping:
            print(
                f"[DelayedEarlyStopping] step {state.global_step} < "
                f"{self.min_steps_before_stopping}: eval_wer not yet considered "
                f"for early stopping (still inside the CTC warm-up window)."
            )
            return control
        return super().on_evaluate(args, state, control, metrics=metrics, **kwargs)


early_stopping_callback = DelayedEarlyStoppingCallback(
    min_steps_before_stopping=MIN_STEPS_BEFORE_EARLY_STOPPING,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)

## Training arguments

`warmup_ratio` replaces the fixed `warmup_steps`, `lr_scheduler_type` is
explicit (`cosine`), `weight_decay` is added, `group_by_length` is on
(matched to the `input_length` column created during preprocessing), and
`fp16` is enabled for the 6-8GB VRAM budget.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    num_train_epochs=NUM_EPOCHS,
    fp16=FP16,
    group_by_length=GROUP_BY_LENGTH,
    length_column_name="input_length",
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=100,
    report_to=[],
    dataloader_num_workers=2,
    seed=SEED,
)

print(training_args)

## Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_proc,
    eval_dataset=val_proc_train,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=processor,
    callbacks=[early_stopping_callback],
)

## Sanity check before committing to the full run

Before launching a multi-day training run, it's worth confirming the
model is not still in a degenerate state at initialization. This runs a
single evaluation pass on the small `val_proc_train` subset with the
*untrained* model, just to confirm the pipeline produces sane (if bad)
output before training starts. Expect a very high WER here — that's
normal and expected for an untrained/pretrained-only checkpoint.


In [ ]:
sanity_metrics = trainer.evaluate(eval_dataset=val_proc_train)
print("Pre-training sanity check (expect a high WER — this is the untrained starting point):")
sanity_metrics

## Train

Full run: `SMOKE_TEST = False`, full `train` split, up to `NUM_EPOCHS`
epochs, with early stopping now correctly delayed past the expected
blank-collapse window. Given the "time isn't a constraint" preference,
this is left to run to completion or until early stopping genuinely
triggers after real convergence — watch that `eval_wer` actually starts
decreasing (not just staying flat) once you're past step
`MIN_STEPS_BEFORE_EARLY_STOPPING`.


In [ ]:
trainer.train()

## Training curves

Same three-panel plot as before: training loss, eval loss, eval WER.
The key thing to check this time: does `eval_wer` actually move (not a
flat line) once training passes `MIN_STEPS_BEFORE_EARLY_STOPPING`? If it's
still flat well past that point, that's a sign something else needs
attention (LR too low now, or a genuine data/label issue) rather than
early stopping being the culprit.


In [ ]:
history_df = pd.DataFrame(trainer.state.log_history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if "loss" in history_df:
    history_df.dropna(subset=["loss"]).plot(x="step", y="loss", ax=axes[0], legend=False)
    axes[0].set_title("Training loss")

if "eval_loss" in history_df:
    history_df.dropna(subset=["eval_loss"]).plot(x="step", y="eval_loss", ax=axes[1], legend=False, color="darkorange")
    axes[1].set_title("Eval loss")

if "eval_wer" in history_df:
    history_df.dropna(subset=["eval_wer"]).plot(x="step", y="eval_wer", ax=axes[2], legend=False, color="seagreen")
    axes[2].axvline(x=MIN_STEPS_BEFORE_EARLY_STOPPING, color="red", linestyle="--", linewidth=1,
                     label="early-stopping guard ends")
    axes[2].set_title("Eval WER")
    axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Final validation metrics

Runs on `val_proc_full` — all of validation, filtered only for CTC
validity — directly comparable to notebook 04's 58.4% baseline corpus
WER on the same split.


In [ ]:
final_metrics = trainer.evaluate(eval_dataset=val_proc_full)
final_metrics

## Training summary

In [ ]:
print("epochs trained:", trainer.state.epoch)
print("best checkpoint:", trainer.state.best_model_checkpoint)
print("best eval WER during training:", trainer.state.best_metric)
print("final full-validation WER:", final_metrics.get("eval_wer"))
print("final full-validation CER:", final_metrics.get("eval_cer"))
print("model saved to:", OUTPUT_DIR)

with open(OUTPUT_DIR / "training_config.json", "w") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "lr": LR,
        "warmup_ratio": WARMUP_RATIO,
        "lr_scheduler_type": LR_SCHEDULER_TYPE,
        "num_epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "gradient_checkpointing": GRADIENT_CHECKPOINTING,
        "fp16": FP16,
        "group_by_length": GROUP_BY_LENGTH,
        "weight_decay": WEIGHT_DECAY,
        "freeze_feature_encoder": FREEZE_FEATURE_ENCODER,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "min_steps_before_early_stopping": MIN_STEPS_BEFORE_EARLY_STOPPING,
        "eval_subset_size_during_training": EVAL_SUBSET_SIZE_DURING_TRAINING,
        "max_audio_duration_sec": MAX_AUDIO_DURATION_SEC,
        "smoke_test": SMOKE_TEST,
        "seed": SEED,
        "final_full_val_wer": final_metrics.get("eval_wer"),
        "final_full_val_cer": final_metrics.get("eval_cer"),
    }, f, indent=2)

print("training_config.json written.")

## Conclusion

Fine-tuning is complete and the best checkpoint (selected by validation
WER, with early stopping correctly delayed past the blank-collapse
window this time) is saved to `models/wav2vec2_slurp_debug/`, along with
the processor and a `training_config.json` recording exactly how this
run was configured.

**How to tell if the fix actually worked:** look at `best eval WER during
training` above. If it's meaningfully below ~0.90-0.95 (rather than
frozen near 0.956 again), `eval_wer` genuinely moved this time, which
means the blank-collapse/early-stopping bug is fixed. If it's still
flat, re-check the training curve plot above — the eval WER should show
real variation once past the `MIN_STEPS_BEFORE_EARLY_STOPPING` line.

Next notebook: `06_finetuned_asr_evaluation.ipynb`. It should load this
checkpoint and reuse the exact same evaluation pipeline as notebook 04,
`src.evaluation.metrics` (`corpus_wer_cer`, `corpus_wer_per_group`),
against the same validation split, so the baseline-vs-fine-tuned
comparison stays apples to apples.

**Possible next experiments** (run independently, one variable at a time,
per the earlier tuning notes):
- `05b`: `FREEZE_FEATURE_ENCODER = False`, everything else identical to
  this run's best checkpoint config — test whether unfreezing the
  encoder helps once the training-dynamics bug is out of the way.
- Data augmentation (background noise, time masking, gain perturbation)
  as a robustness follow-up once a solid fine-tuned baseline exists.
